# Projeto Final, Assistente Educacional Multi-AgenteNotebook executável que acompanha os guias do Módulo 14([01-arquitetura.md](../../lessons/modulo-14-projeto-final/01-arquitetura.md) e [02-construcao-e-avaliacao.md](../../lessons/modulo-14-projeto-final/02-construcao-e-avaliacao.md)).## O que vamos fazer aquiVamos conversar com o assistente educacional completo, que integra tudo o que a trilha construiu:RAG por trecho com citação (M9), ferramenta de cálculo e roteamento do agente (M10), time de agentescoordenados (M11), Learning Analytics com relatório (M12) e modelagem do aluno com knowledgetracing, pré-requisitos e personalização (M13). Importamos o assistente do projeto e o conduzimospor uma sessão real, sempre em mensagens de texto livre. Python puro.

## Carregando o assistente do projetoO código completo e comentado está em `projects/m14-final-assistant/`. Aqui o importamos e montamoso assistente com a base de conhecimento de exemplo e um perfil de aluna que prefere exemplosvisuais. A partir daí, é só usar `assistente.responder(...)` com a mensagem do aluno, que o roteadordecide o que fazer.

In [ ]:
import sys
import os

# Adiciona o projeto ao caminho de importação (a partir da pasta deste notebook).
projeto = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "projects", "m14-final-assistant"))
if projeto not in sys.path:
    sys.path.insert(0, projeto)

from final_assistant import (
    AssistenteEducacional, ModeloAluno, proximo_tema, sugestao_mentor, TEMAS,
)

aluna = ModeloAluno("Ana", preferencias=["exemplos visuais"])
assistente = AssistenteEducacional(modelo=aluna)
responder = assistente.responder
print("Assistente pronto. Temas da trilha:", TEMAS)

## O Tutor busca o trecho e cita a fonteA primeira coisa que um aluno faz é perguntar. O Tutor usa o RAG do Módulo 9 para recuperar o trechomais relevante do material e responde citando a fonte. Quando a pergunta está fora do material, eleadmite com honestidade, em vez de inventar.

In [ ]:
print(responder("o que é a derivada?"))
print()
print(responder("qual é a capital da França?"))   # fora do material

Repare em dois pontos. A explicação vem marcada como **detalhada, nível iniciante, com um diagrama**,porque o domínio da aluna naquele tema ainda é baixo e o perfil dela prefere exemplos visuais, e aresposta traz a **fonte** entre parênteses. Já a pergunta fora do material recebe um honesto nãoencontrei, o tratamento da ausência que vimos no Módulo 9.

## A calculadora e o ciclo de exercíciosO aluno também pode pedir uma conta ou um exercício, na mesma conversa. O roteador reconhece aintenção. O assistente propõe o exercício na dificuldade recomendada, o aluno responde, e o Evaluatorcorrige e atualiza o domínio do tema com knowledge tracing.

In [ ]:
print(responder("28*3/4"))
print(responder("quero um exercício de derivada"))
print(responder("6"))                 # responde o exercício proposto
print(responder("exercício de derivada"))
print(responder("3"))                 # acerta de novo, o domínio sobe mais
print("domínio de derivada agora:", assistente.modelo.dominio["derivada"])

A cada acerto, o knowledge tracing eleva a estimativa de domínio da aluna em derivada. Esse estado écompartilhado: é ele que vai mudar, logo abaixo, a forma como o Tutor explica o mesmo tema.

## A profundidade se adapta ao domínioAqui está o coração da personalização. A mesma pergunta do início, feita agora que o domínio subiu,recebe uma explicação diferente, mais breve. Não mudamos nada no Tutor, apenas o modelo da alunaevoluiu, e a resposta acompanhou.

In [ ]:
print(responder("o que é a derivada?"))

A explicação que antes era detalhada agora vem **breve, nível avançado**. É a prova de que aintegração entregou mais do que a soma das partes: a avaliação alimentou o modelo, e o modelo mudoua explicação.

## Pré-requisitos e o próximo temaO Mentor não recomenda qualquer tema. Ele respeita um grafo de pré-requisitos: o determinante, porexemplo, só é sugerido depois que a matriz está dominada. Vamos fazer a aluna errar um exercício dematriz e ver o que o sistema recomenda em seguida.

In [ ]:
print(responder("exercício de matriz"))
print(responder("99"))                 # erra, o domínio de matriz cai
print("temas fracos:", assistente.modelo.temas_fracos())
print("próximo tema recomendado:", proximo_tema(assistente.modelo))

Como a matriz ficou fraca, ela volta como prioridade, e o determinante, que depende dela, continuabloqueado. É o ensino seguindo uma ordem que faz sentido, não uma lista solta de temas.

## O relatório da sessãoAo final, o Analytics do Módulo 12 resume a sessão: engajamento, domínio por tema em barras, risco dedesengajamento e a orientação do Mentor. Numa sessão de verdade, esse relatório é gravado em`relatorio_sessao.md`, e o modelo da aluna é salvo para ser recarregado na próxima, garantindo amemória de longo prazo.

In [ ]:
print(responder("relatório"))
print()
print(assistente.analytics.relatorio_markdown(
    assistente.modelo, sugestao_mentor(assistente.modelo, assistente.analytics)))

## Conclusão da trilhaSe você chegou até aqui, percorreu o caminho completo, do que é IA até um assistente educacionalinteligente, multi-agente, com analytics e modelagem de longo prazo do aluno, entendendo cada peçapor dentro, do tokenizador ao knowledge tracing. Esse entendimento, sem caixas-pretas, é o que vaite permitir pesquisar, melhorar e criar os assistentes educacionais do futuro.Para estender o assistente, troque a busca por embeddings densos e um banco vetorial, ative a geraçãovia Ollama para o Tutor redigir a explicação na profundidade recomendada, e construa o dashboard doMódulo 12 sobre os dados de analytics. Cada extensão é plugar uma peça mais sofisticada de um módulono lugar de uma simples. Bom trabalho!